In [ ]:
# 1) GPU check + package installs
import importlib
import sys

print(sys.version)

!nvidia-smi
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q pretty_midi mido tqdm

# Verify the packages are importable in this kernel.
for module_name in ['pretty_midi', 'mido', 'tqdm']:
    importlib.import_module(module_name)
print('Package check passed: pretty_midi, mido, tqdm')

In [ ]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false
# 2) All imports + config values (Task 4 RLHF settings)
import csv
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

SEED = 42

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
VALIDATION_SPLIT = 0.2
NUM_WORKERS = 0

TOKEN_PAD_ID = 0
TOKEN_BOS_ID = 1
TOKEN_EOS_ID = 2
TOKEN_NOTE_OFFSET = 3
TOKEN_SEQUENCE_LENGTH = 512
TOKEN_VOCAB_SIZE = 91

TR_MODEL_DIM = 256
TR_NUM_HEADS = 8
TR_NUM_LAYERS = 6
TR_FF_DIM = 1024
TR_DROPOUT = 0.1
TR_LABEL_SMOOTHING = 0.05
TR_NUM_SAMPLES = 10
TR_GENERATION_MAX_TOKENS = 512

RLHF_NUM_EPOCHS = 10
RLHF_SAMPLES_PER_EPOCH = 16
RLHF_LEARNING_RATE = 1e-5
RLHF_WEIGHT_DECAY = 1e-5
RLHF_TEMPERATURE = 1.0
RLHF_TOP_K = 16
RLHF_EVALUATION_SAMPLES = 10
RLHF_NUM_SAMPLES = 10
RLHF_MAX_NEW_TOKENS = 512

OUTPUT_ROOT = Path('/kaggle/working/outputs')
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
GENERATED_MIDI_DIR = OUTPUT_ROOT / 'generated_midis'
SURVEY_RESULTS_DIR = OUTPUT_ROOT / 'survey_results'

for directory in [OUTPUT_ROOT, CHECKPOINTS_DIR, PLOTS_DIR, GENERATED_MIDI_DIR, SURVEY_RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CODE_INPUT_CANDIDATES = [
    Path('/kaggle/input/music-generation-unsupervised-task4-src'),
    Path('/kaggle/input/music-generation-unsupervised-task3-src'),
    Path('/kaggle/input/music-generation-unsupervised-task2-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task4-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task3-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task2-src'),
]

code_input_root = None
for candidate in CODE_INPUT_CANDIDATES:
    if candidate.exists():
        code_input_root = candidate
        break

if code_input_root is None:
    raise FileNotFoundError(
        'Code dataset path not found. Run "!ls /kaggle/input" and update CODE_INPUT_CANDIDATES.'
    )

if (code_input_root / 'music-generation-unsupervised').exists():
    code_input_root = code_input_root / 'music-generation-unsupervised'

PROJECT_ROOT = Path('/kaggle/working/music-generation-unsupervised')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(code_input_root, PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Code input used:', code_input_root)
print('Project root exists:', PROJECT_ROOT.exists())

In [ ]:
# pyright: reportMissingImports=false
# 3) Data loading and preprocessing (Task 3 checkpoint + survey scores)
TASK3_CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / 'outputs' / 'checkpoints' / 'task3_best_transformer.pt',
    CHECKPOINTS_DIR / 'task3_best_transformer.pt',
    code_input_root / 'outputs' / 'checkpoints' / 'task3_best_transformer.pt',
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task4-src/music-generation-unsupervised/outputs/checkpoints/task3_best_transformer.pt'),
    Path('/kaggle/input/music-generation-unsupervised-task4-src/outputs/checkpoints/task3_best_transformer.pt'),
    Path('/kaggle/input/music-generation-unsupervised-task3-outputs/checkpoints/task3_best_transformer.pt'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task3-outputs/checkpoints/task3_best_transformer.pt'),
]

BASE_CHECKPOINT_PATH = None
for candidate in TASK3_CHECKPOINT_CANDIDATES:
    if candidate.exists():
        BASE_CHECKPOINT_PATH = candidate
        break

if BASE_CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        'Task 3 checkpoint not found. Include task3_best_transformer.pt in outputs/checkpoints of the Task 4 code dataset, or attach a Task 3 outputs dataset.'
    )

SURVEY_SCORE_CANDIDATES = [
    PROJECT_ROOT / 'outputs' / 'survey_results' / 'scores.csv',
    code_input_root / 'outputs' / 'survey_results' / 'scores.csv',
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task4-src/music-generation-unsupervised/outputs/survey_results/scores.csv'),
    Path('/kaggle/input/music-generation-unsupervised-task4-src/outputs/survey_results/scores.csv'),
    Path('/kaggle/input/music-generation-unsupervised-task4-survey/outputs/survey_results/scores.csv'),
    Path('/kaggle/input/music-generation-unsupervised-task4-survey/scores.csv'),
    SURVEY_RESULTS_DIR / 'scores.csv',
]

SURVEY_SCORES_PATH = None
for candidate in SURVEY_SCORE_CANDIDATES:
    if candidate.exists():
        SURVEY_SCORES_PATH = candidate
        break

if SURVEY_SCORES_PATH is None:
    SURVEY_SCORES_PATH = SURVEY_RESULTS_DIR / 'scores.csv'
    with SURVEY_SCORES_PATH.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=['participant_id', 'sample_id', 'midi_path', 'score'],
        )
        writer.writeheader()
        for participant_index in range(1, 11):
            for sample_index in range(1, 11):
                writer.writerow(
                    {
                        'participant_id': f'participant_{participant_index}',
                        'sample_id': f'task3_sample_{sample_index}',
                        'midi_path': str(GENERATED_MIDI_DIR / f'task3_sample_{sample_index}.mid'),
                        'score': '',
                    }
                )

print('Base checkpoint:', BASE_CHECKPOINT_PATH)
print('Survey scores CSV:', SURVEY_SCORES_PATH)
print('scores.csv can contain simulated ratings for pipeline testing, or real ratings (1-5) for final RLHF.')

In [ ]:
# 4) Model definition (load Task 3 base Transformer without holding extra GPU copy)
from src.generation.generate_music import load_trained_transformer

base_model = load_trained_transformer(
    checkpoint_path=BASE_CHECKPOINT_PATH,
    device=torch.device('cpu'),
)

num_parameters = sum(param.numel() for param in base_model.parameters())
print(f'Task 3 base model loaded on CPU with {num_parameters:,} parameters.')

del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 5) Training loop (from src/training/rlhf_finetune.py)
from src.training.rlhf_finetune import rlhf_finetune_transformer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    results = rlhf_finetune_transformer(
        base_checkpoint_path=BASE_CHECKPOINT_PATH,
        survey_csv_path=SURVEY_SCORES_PATH,
        num_epochs=RLHF_NUM_EPOCHS,
        samples_per_epoch=RLHF_SAMPLES_PER_EPOCH,
        learning_rate=RLHF_LEARNING_RATE,
        weight_decay=RLHF_WEIGHT_DECAY,
        max_new_tokens=RLHF_MAX_NEW_TOKENS,
        temperature=RLHF_TEMPERATURE,
        top_k=RLHF_TOP_K,
        evaluation_samples=RLHF_EVALUATION_SAMPLES,
    )
except torch.OutOfMemoryError:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    fallback_samples_per_epoch = min(RLHF_SAMPLES_PER_EPOCH, 8)
    fallback_max_new_tokens = min(RLHF_MAX_NEW_TOKENS, 256)
    fallback_eval_samples = min(RLHF_EVALUATION_SAMPLES, 5)

    print('CUDA OOM with default Task 4 settings. Retrying with fallback values:')
    print('samples_per_epoch =', fallback_samples_per_epoch)
    print('max_new_tokens =', fallback_max_new_tokens)
    print('evaluation_samples =', fallback_eval_samples)

    results = rlhf_finetune_transformer(
        base_checkpoint_path=BASE_CHECKPOINT_PATH,
        survey_csv_path=SURVEY_SCORES_PATH,
        num_epochs=RLHF_NUM_EPOCHS,
        samples_per_epoch=fallback_samples_per_epoch,
        learning_rate=RLHF_LEARNING_RATE,
        weight_decay=RLHF_WEIGHT_DECAY,
        max_new_tokens=fallback_max_new_tokens,
        temperature=RLHF_TEMPERATURE,
        top_k=RLHF_TOP_K,
        evaluation_samples=fallback_eval_samples,
    )

print('Best checkpoint saved at:', results['checkpoint_path'])
print('Comparison plot saved at:', results['comparison_plot_path'])
print('Survey rows:', results['survey_rows'])
print('Valid scores:', results['valid_scores'])
print('Participants:', results['participants'])

In [ ]:
# 6) Loss / metric plots — save to /kaggle/working/outputs/plots/
comparison_plot_path = Path(results['comparison_plot_path'])
reward_curve_path = PLOTS_DIR / 'task4_reward_curve.png'

plt.figure(figsize=(10, 5))
plt.plot(results['history_rewards'], label='Mean Reward')
plt.xlabel('Epoch')
plt.ylabel('Reward')
plt.title('Task 4 - RLHF Reward Curve')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(reward_curve_path, dpi=200)
plt.show()

comparison_image = plt.imread(comparison_plot_path)
plt.figure(figsize=(10, 5))
plt.imshow(comparison_image)
plt.axis('off')
plt.title('Task 4 - Before vs After RLHF')
plt.tight_layout()
plt.show()

print('Saved:', comparison_plot_path)
print('Saved:', reward_curve_path)

In [ ]:
# 7) Music generation — save MIDIs to /kaggle/working/outputs/generated_midis/
from src.generation.generate_music import generate_task4_samples

generated_paths = generate_task4_samples(
    checkpoint_path=results['checkpoint_path'],
    num_samples=RLHF_NUM_SAMPLES,
    max_new_tokens=RLHF_MAX_NEW_TOKENS,
    output_dir=GENERATED_MIDI_DIR,
)

for path in generated_paths:
    print(path)

In [ ]:
# 8) Summary cell printing all deliverables
expected_files = [
    SURVEY_SCORES_PATH,
    PLOTS_DIR / 'task4_before_after_comparison.png',
] + [GENERATED_MIDI_DIR / f'task4_sample_{index}.mid' for index in range(1, 11)]

print('Task 4 Deliverables Summary')
for file_path in expected_files:
    status = 'OK' if file_path.exists() else 'MISSING'
    print(f'[{status}] {file_path}')